In [1]:
# Select embedding model
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("/Users/zhaojie/project/langgraphtest0725/model_files/embeddingmodel/all-mpnet-base-v2")
print(embedding_model.encode("你好哇 先生"))

/opt/miniconda3/envs/langgraph-test-0725/lib/python3.12/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/opt/miniconda3/envs/langgraph-test-0725/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[ 3.13723013e-02 -8.15289654e-03 -1.15727317e-02  1.07067619e-02
  1.39965797e-02 -1.22439405e-02 -6.50949702e-02  3.47239859e-02
  1.05272690e-02  2.43837722e-02  1.06491968e-02 -3.52041498e-02
  5.27302772e-02  2.41654310e-02 -4.65646461e-02 -6.45199642e-02
  1.85884684e-02  2.73917112e-02  4.58659306e-02 -1.85253751e-02
 -3.72554064e-02  1.06794676e-02  2.67252252e-02  1.60255320e-02
  3.61067168e-02 -4.81615067e-02  1.39088342e-02 -2.20511984e-02
  4.48988006e-03  1.80214681e-02  2.27014404e-02 -2.42200512e-02
 -4.18279767e-02  1.75161322e-03  1.62636366e-06 -1.03442138e-02
 -8.53871182e-03 -3.41529809e-02 -2.62655038e-02 -4.49964590e-02
  2.41143182e-02 -2.32828129e-02 -2.06837878e-02  1.98148619e-02
 -1.28621776e-02 -3.13797854e-02  1.86386090e-02  4.71481308e-03
  2.22410802e-02  1.86205599e-02 -8.40432476e-04 -9.33283865e-02
 -1.65852010e-02 -1.36640258e-02  3.13267298e-02  2.28635129e-02
  1.56473671e-03 -3.86138111e-02 -6.64851665e-02  2.83993781e-02
 -8.37120414e-03 -4.01876

In [6]:
# 2. Build and train an intent classification model
import numpy as np
from sklearn.base import TransformerMixin, BaseEstimator
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from tqdm.notebook import tqdm

# Note: This is a very small dataset.
# More data will help make the model  more accurate and avoid overfitting.
sample_data = {
    "text": [
        # Greeting utterances
        "hi",
        "hello",
        "howdy",
        "hey there",
        "greetings",
        "Nice to see you",
        "Let's start",
        "begin",
        "good morning",
        "Good afternoon",
        # Menu utterances
        "I want to talk about something else",
        "options",
        "menu, please",
        "Could we chat about another subject",
        "I want to see the menu",
        "switch topics",
        "What else can you do",
        "discuss about something else",
        "Show me the menu",
        "Can we do something else",
        # Restart utterances
        "restart",
        "I'd like to do this again",
        "let me try again",
        "one more time",
        "Can I review that?",
        "check again",
        "redo",
        "again please",
        "that was great, let's start from the beginning",
        "go back to start",
    ],
    "intent": [
        "greeting",
        "greeting",
        "greeting",
        "greeting",
        "greeting",
        "greeting",
        "greeting",
        "greeting",
        "greeting",
        "greeting",
        "menu",
        "menu",
        "menu",
        "menu",
        "menu",
        "menu",
        "menu",
        "menu",
        "menu",
        "menu",
        "restart",
        "restart",
        "restart",
        "restart",
        "restart",
        "restart",
        "restart",
        "restart",
        "restart",
        "restart",
    ]
}
print(len(sample_data["text"]),len(sample_data["intent"]))
import pandas as pd

df = pd.DataFrame(sample_data)
df.head()
X_train, X_test, y_train, y_test = train_test_split(
    df["text"],
    df["intent"],
    test_size=0.5,
    random_state=14
)


class Encoder(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.embedding_model = embedding_model

    def transform(self, X):
        return self.embedding_model.encode(list(X))

    def fit(self, X, y=None):
        return self


pipeline = Pipeline([
    ('encoder', Encoder()),
    ('clf', LogisticRegression()),
])
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)
# y_pred
single_pred = pipeline.predict(["Please let's move on"])
# single_pred
probas = pipeline.predict_proba(["Please let's move on"])
# probas
confidence_score = float(np.max(probas, axis=1)[0])
# confidence_score
print("\nClassification Report:\n", classification_report(y_test, y_pred))

30 30

Classification Report:
               precision    recall  f1-score   support

    greeting       0.83      1.00      0.91         5
        menu       0.67      1.00      0.80         4
     restart       1.00      0.50      0.67         6

    accuracy                           0.80        15
   macro avg       0.83      0.83      0.79        15
weighted avg       0.86      0.80      0.78        15

